<a href="https://colab.research.google.com/github/oseiakotokwarteng-R-Insights/causal-inference-portfolio/blob/main/organ_donation_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Hello from R")

[1] "Hello from R"


In [2]:
install.packages("causaldata")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [3]:
install.packages("tidyverse")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [4]:
library(causaldata)
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [5]:
data("organ_donations")

In [7]:
nrow(organ_donations)

[1] 162

In [8]:
length(unique(organ_donations$State))

[1] 27

In [9]:
unique(organ_donations$Quarter)

[1] "Q42010" "Q12011" "Q22011" "Q32011" "Q42011" "Q12012"

In [10]:
summary(organ_donations$Rate)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1229  0.3346  0.4397  0.4450  0.5645  0.7900 

In [13]:
organ_donations %>%
  filter(State == "California") %>%
  summarize(min = min(Rate), max = max(Rate), mean = mean(Rate), median = median(Rate))

min,max,mean,median
<dbl>,<dbl>,<dbl>,<dbl>
0.2607,0.2743,0.2670667,0.26535


In [12]:
unique(organ_donations$State)

[1] "Alaska"               "Arizona"              "California"          
 [4] "Colorado"             "Connecticut"          "District of Columbia"
 [7] "Florida"              "Hawaii"               "Louisiana"           
[10] "Maryland"             "Michigan"             "Minnesota"           
[13] "Missouri"             "Montana"              "Nebraska"            
[16] "New Hampshire"        "New Jersey"           "New York"            
[19] "North Carolina"       "Ohio"                 "Pennsylvania"        
[22] "South Carolina"       "Tennessee"            "Virginia"            
[25] "Washington"           "Wisconsin"            "Wyoming"

In [14]:
organ_donations %>%
  filter(State %in% c("Arizona", "New York", "Ohio")) %>%
  arrange(State, Quarter_Num)

State,Quarter,Rate,Quarter_Num
<chr>,<chr>,<dbl>,<int>
Arizona,Q42010,0.2634,1
Arizona,Q12011,0.2092,2
Arizona,Q22011,0.2261,3
Arizona,Q32011,0.2503,4
Arizona,Q42011,0.2351,5
Arizona,Q12012,0.2587,6
New York,Q42010,0.1249,1
New York,Q12011,0.1267,2
New York,Q22011,0.1314,3


In [15]:
organ_donations %>%
  filter(State %in% c("Arizona", "New York", "Ohio", "California")) %>%
  arrange(State, Quarter_Num)

State,Quarter,Rate,Quarter_Num
<chr>,<chr>,<dbl>,<int>
Arizona,Q42010,0.2634,1
Arizona,Q12011,0.2092,2
Arizona,Q22011,0.2261,3
Arizona,Q32011,0.2503,4
Arizona,Q42011,0.2351,5
Arizona,Q12012,0.2587,6
California,Q42010,0.2666,1
California,Q12011,0.2731,2
California,Q22011,0.2743,3


In [16]:
organ_donations %>%
  arrange(State, Quarter_Num)

State,Quarter,Rate,Quarter_Num
<chr>,<chr>,<dbl>,<int>
Alaska,Q42010,0.7500,1
Alaska,Q12011,0.7700,2
Alaska,Q22011,0.7700,3
Alaska,Q32011,0.7800,4
Alaska,Q42011,0.7800,5
Alaska,Q12012,0.7900,6
Arizona,Q42010,0.2634,1
Arizona,Q12011,0.2092,2
Arizona,Q22011,0.2261,3


In [23]:
organ_donations %>%
  filter(Quarter_Num %in% c(3,4)) %>%
  select(State, Quarter_Num, Rate) %>%
  pivot_wider(names_from = Quarter_Num, values_from = Rate, names_prefix = "Q") %>%
  mutate(change = Q4 - Q3) %>%
  arrange(change)

State,Q3,Q4,change
<chr>,<dbl>,<dbl>,<dbl>
New Hampshire,0.5643,0.5334,-0.0309
California,0.2743,0.2636,-0.0107
Maryland,0.4679,0.4590,-0.0089
Washington,0.5882,0.5846,-0.0036
Wisconsin,0.5720,0.5688,-0.0032
Wyoming,0.5937,0.5911,-0.0026
Colorado,0.6736,0.6715,-0.0021
Ohio,0.5701,0.5701,0.0000
Pennsylvania,0.4553,0.4559,0.0006


In [24]:
organ_donations %>%
  mutate(period = ifelse(Quarter_Num <= 3, "pre", "post")) %>%
  group_by(State, period) %>%
  summarize(avg_rate = mean(Rate), .groups = "drop") %>%
  pivot_wider(names_from = period, values_from = avg_rate) %>%
  mutate (change = post-pre) %>%
  arrange(change)

State,post,pre,change
<chr>,<dbl>,<dbl>,<dbl>
New Hampshire,0.5278333,0.5604333,-3.260000e-02
South Carolina,0.2629000,0.2752000,-1.230000e-02
California,0.2628000,0.2713333,-8.533333e-03
Wyoming,0.5882000,0.5938000,-5.600000e-03
Colorado,0.6669333,0.6687667,-1.833333e-03
Wisconsin,0.5720667,0.5721667,-1.000000e-04
North Carolina,0.5250000,0.5249667,3.333333e-05
Pennsylvania,0.4545667,0.4540000,5.666667e-04
Washington,0.5878333,0.5862000,1.633333e-03


In [25]:
did_table <- organ_donations %>%
  mutate(period = ifelse(Quarter_Num <= 3, "pre", "post")) %>%
  group_by(State, period) %>%
  summarize(avg_rate = mean(Rate), .groups = "drop") %>%
  pivot_wider(names_from = period, values_from = avg_rate) %>%
  mutate (change = post-pre)

In [26]:
ca_change <- did_table %>%
  filter(State == "California") %>%
  pull(change)


In [27]:
comparison_change <- did_table %>%
  filter(State != "California") %>%
  summarize(avg_change = mean(change)) %>%
  pull(avg_change)

In [28]:
did_estimate <- ca_change - comparison_change

In [29]:
ca_change
comparison_change
did_estimate

[1] -0.008533333

[1] 0.01392564

[1] -0.02245897